# LeetCode 41 — First Missing Positive (Hard) bằng Python

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def firstMissingPositive(self, nums: List[int]) -> int:

```

**Mục tiêu học:**

1. Hiểu đúng đề: "số nguyên dương nhỏ nhất **không** có trong mảng" là gì, kể cả khi có số âm, số 0, số trùng, số rất lớn.
2. Đi từ cách trực giác (thử 1, 2, 3, ...) → sắp xếp → dùng `set` → hai cách **O(n) thời gian, O(1) bộ nhớ phụ**.
3. Nắm **nhận xét then chốt**: đáp án luôn nằm trong đoạn `[1, n + 1]`.
4. Học kỹ thuật **dùng chính mảng làm bảng đánh dấu** (index-as-hash): đánh dấu bằng dấu âm, và **cyclic sort** (đưa mỗi số `x` về "nhà" của nó ở chỉ số `x - 1`).
5. Hiểu vì sao cyclic sort chỉ tốn tối đa `n` lần swap dù có vòng `while` lồng trong `for`.
6. Tự viết lại lời giải, tránh các bẫy kinh điển (swap một dòng sai thứ tự, vòng lặp vô hạn khi có số trùng, lật dấu hai lần).

## 1. Đề bài (theo LeetCode)

Cho một mảng số nguyên **chưa sắp xếp** `nums`. Trả về **số nguyên dương nhỏ nhất không xuất hiện** trong `nums`.

Bạn phải cài đặt thuật toán chạy trong thời gian **O(n)** và dùng **O(1) bộ nhớ phụ** (auxiliary space).

**Ví dụ 1**

```text
Input:  nums = [1,2,0]
Output: 3
Giải thích: các số trong đoạn [1,2] đều có trong mảng.
```

**Ví dụ 2**

```text
Input:  nums = [3,4,-1,1]
Output: 2
Giải thích: 1 có trong mảng nhưng 2 bị thiếu.
```

**Ví dụ 3**

```text
Input:  nums = [7,8,9,11,12]
Output: 1
Giải thích: số nguyên dương nhỏ nhất là 1 đã bị thiếu.
```

**Constraints (ràng buộc):**

- `1 <= nums.length <= 10^5`
- `-2^31 <= nums[i] <= 2^31 - 1`

### Đọc đề cho kỹ

- **Số nguyên dương** là `1, 2, 3, ...`. Số `0` **không** phải số dương.
- Mảng có thể chứa số âm, số 0, số **trùng nhau**, và số rất lớn (tới khoảng 2 tỷ).
- Ta không cần trả về vị trí, chỉ cần trả về **một con số**.
- "O(1) bộ nhớ phụ" nghĩa là: ngoài mảng `nums` đề đưa, ta chỉ được dùng vài biến lẻ (`i`, `j`, `n`, ...). **Không** được tạo thêm list/set/dict có kích thước tăng theo `n`. Nhưng ta **được phép sửa trực tiếp** mảng `nums` (gọi là **in-place** — "tại chỗ").

Cách hiểu nhanh nhất: lần lượt hỏi "có 1 không? có 2 không? có 3 không?..." — câu "không" đầu tiên chính là đáp án.

Thử nhìn các ví dụ bằng code: ta lọc riêng các số dương ra xem.

In [1]:
from typing import List
import random

examples = [
    ([1, 2, 0], 3),
    ([3, 4, -1, 1], 2),
    ([7, 8, 9, 11, 12], 1),
]

for nums, expected in examples:
    positives = sorted(x for x in nums if x > 0)
    print(f"nums = {nums!s:<18} số dương (đã sắp xếp) = {positives!s:<18} đáp án = {expected}")

nums = [1, 2, 0]          số dương (đã sắp xếp) = [1, 2]             đáp án = 3
nums = [3, 4, -1, 1]      số dương (đã sắp xếp) = [1, 3, 4]          đáp án = 2
nums = [7, 8, 9, 11, 12]  số dương (đã sắp xếp) = [7, 8, 9, 11, 12]  đáp án = 1


Nhìn cột "số dương đã sắp xếp":

- `[1, 2]` → có 1, có 2, thiếu 3 → đáp án `3`.
- `[1, 3, 4]` → có 1, thiếu 2 → đáp án `2`.
- `[7, 8, 9, 11, 12]` → thiếu ngay 1 → đáp án `1`.

Số âm và số 0 **không bao giờ** là đáp án và cũng không giúp "lấp chỗ trống" nào. Ta sẽ thấy ở mục 6 rằng cả các số **quá lớn** cũng vô dụng.

## 2. Kiến thức nền 1 — toán tử `in` trên list tốn bao nhiêu?

`x in nums` hỏi: "giá trị `x` có nằm trong list không?". Python làm việc này bằng cách **đi lần lượt từ đầu tới cuối** list và so sánh từng phần tử. Nếu không tìm thấy, nó phải so sánh **hết n phần tử** → mỗi lần `in` tốn **O(n)**.

Ta tự viết lại `in` để đếm số lần so sánh:

In [2]:
def contains_count(nums, x):
    """Giống `x in nums`, nhưng trả thêm số lần so sánh."""
    comparisons = 0
    for value in nums:
        comparisons += 1
        if value == x:
            return True, comparisons
    return False, comparisons

nums = [3, 4, -1, 1]
print(1 in nums, 2 in nums)
print("tìm 1:", contains_count(nums, 1))   # thấy ở cuối -> 4 lần so sánh
print("tìm 2:", contains_count(nums, 2))   # không có  -> phải so hết 4 phần tử

True False
tìm 1: (True, 4)
tìm 2: (False, 4)


## 3. Cách 1 — Thử lần lượt 1, 2, 3, ... (Brute Force)

Đây đúng là cách ta làm bằng tay:

```text
k = 1
trong khi k có trong nums:
    k = k + 1
trả về k
```

Vòng `while` chắc chắn dừng vì mảng chỉ có hữu hạn phần tử. Nhưng **dừng sau bao nhiêu lần**? Tạm thời cứ chạy đã, mục 6 sẽ trả lời chính xác.

In [3]:
def first_missing_bruteforce(nums: List[int]) -> int:
    k = 1
    while k in nums:      # mỗi lần `in` là một lần quét list: O(n)
        k += 1
    return k

for nums, expected in examples:
    print(nums, "->", first_missing_bruteforce(nums), "| mong đợi:", expected)

[1, 2, 0] -> 3 | mong đợi: 3
[3, 4, -1, 1] -> 2 | mong đợi: 2
[7, 8, 9, 11, 12] -> 1 | mong đợi: 1


### Cách 1 chậm cỡ nào?

Trường hợp xấu nhất: `nums = [1, 2, ..., n]` (có thể bị xáo trộn). Ta phải hỏi `1 in nums`, `2 in nums`, ..., `n in nums`, rồi `n + 1 in nums` — tổng cộng `n + 1` lần, mỗi lần quét tới `n` phần tử.

Đếm thử số phép so sánh (dùng `contains_count` ở trên). Để đúng "xấu nhất", ta đặt mảng **ngược** `[n, n-1, ..., 1]` để số nhỏ nằm cuối:

In [4]:
def bruteforce_comparisons(nums):
    k, total = 1, 0
    while True:
        found, c = contains_count(nums, k)
        total += c
        if not found:
            return k, total
        k += 1

for n in [10, 100, 1000]:
    nums = list(range(n, 0, -1))          # [n, n-1, ..., 1]
    ans, total = bruteforce_comparisons(nums)
    print(f"n = {n:>5}: đáp án = {ans:>5}, số phép so sánh = {total:>9,}  (~ n²/2 = {n*n//2:,})")

n =    10: đáp án =    11, số phép so sánh =        65  (~ n²/2 = 50)
n =   100: đáp án =   101, số phép so sánh =     5,150  (~ n²/2 = 5,000)
n =  1000: đáp án =  1001, số phép so sánh =   501,500  (~ n²/2 = 500,000)


`n` tăng 10 lần → số phép so sánh tăng khoảng **100 lần**. Đó là dấu hiệu của **O(n²)**.

Với `n = 10^5` (giới hạn đề), ta cần cỡ `5 * 10^9` phép so sánh → chắc chắn **Time Limit Exceeded**.

| | Cách 1 |
|---|---|
| Thời gian | `O(n²)` (tối đa `n + 1` lần `in`, mỗi lần `O(n)`) |
| Bộ nhớ phụ | `O(1)` |

**Vướng ở đâu?** Mỗi lần hỏi "có `k` không?" ta lại quét **lại từ đầu** cả mảng. Ta cần một cách hỏi nhanh hơn, hoặc sắp xếp dữ liệu để khỏi phải hỏi lại.

## 4. Cách 2 — Sắp xếp rồi quét một lượt

Nếu mảng đã **sắp xếp tăng dần**, các số dương đứng thành hàng: `1, 2, 3, ...`. Ta chỉ cần đi một lượt và giữ biến `expected` = "số dương tiếp theo ta đang chờ gặp".

**Kiến thức nền: `sorted()` và `.sort()`**

- `sorted(nums)` tạo **list mới** đã sắp xếp, `nums` giữ nguyên.
- `nums.sort()` sắp xếp **ngay trong** `nums` (in-place), trả về `None`.
- Cả hai tốn `O(n log n)` thời gian.

In [5]:
a = [3, 4, -1, 1]
b = sorted(a)
print("sorted(a) =", b, "| a vẫn là", a)

a.sort()
print("sau a.sort(): a =", a)

sorted(a) = [-1, 1, 3, 4] | a vẫn là [3, 4, -1, 1]
sau a.sort(): a = [-1, 1, 3, 4]


### Quy tắc khi quét mảng đã sắp xếp

Bắt đầu `expected = 1`. Gặp mỗi `x`:

| Trường hợp | Ý nghĩa | Làm gì |
|---|---|---|
| `x < expected` | số âm, số 0, hoặc số **trùng** với số đã gặp | bỏ qua |
| `x == expected` | gặp đúng số đang chờ | `expected += 1` |
| `x > expected` | có "lỗ hổng": `expected` bị nhảy qua | dừng, đáp án là `expected` |

Nếu quét hết mảng mà không dừng, đáp án vẫn là `expected`.

Lưu ý dòng đầu: nhờ `x < expected` mà số trùng tự động bị bỏ qua. Nếu viết `x != expected` thì số trùng sẽ bị hiểu nhầm là "lỗ hổng".

In [6]:
def first_missing_sort(nums: List[int]) -> int:
    expected = 1
    for x in sorted(nums):
        if x == expected:
            expected += 1
        elif x > expected:
            break            # có lỗ hổng, không cần xem tiếp
        # còn lại: x < expected -> bỏ qua
    return expected


def trace_sort(nums):
    s = sorted(nums)
    print("sorted:", s)
    expected = 1
    for x in s:
        if x == expected:
            print(f"  x = {x:>2}: bằng expected -> expected = {expected + 1}")
            expected += 1
        elif x > expected:
            print(f"  x = {x:>2}: lớn hơn expected = {expected} -> DỪNG")
            break
        else:
            print(f"  x = {x:>2}: nhỏ hơn expected = {expected} -> bỏ qua")
    print("Đáp án:", expected)

trace_sort([2, -3, 1, 0, 2, 5, 1])

sorted: [-3, 0, 1, 1, 2, 2, 5]
  x = -3: nhỏ hơn expected = 1 -> bỏ qua
  x =  0: nhỏ hơn expected = 1 -> bỏ qua
  x =  1: bằng expected -> expected = 2
  x =  1: nhỏ hơn expected = 2 -> bỏ qua
  x =  2: bằng expected -> expected = 3
  x =  2: nhỏ hơn expected = 3 -> bỏ qua
  x =  5: lớn hơn expected = 3 -> DỪNG
Đáp án: 3


Trace cho thấy đủ 3 nhánh: số âm/0 bị bỏ qua, số trùng (`1` thứ hai, `2` thứ hai) bị bỏ qua, và `5 > 3` làm vòng lặp dừng → đáp án `3`.

| | Cách 2 |
|---|---|
| Thời gian | `O(n log n)` do sắp xếp, phần quét chỉ `O(n)` |
| Bộ nhớ phụ | `O(n)` với `sorted()` (tạo list mới). Dùng `nums.sort()` thì đỡ hơn, nhưng thuật toán sort của Python (Timsort) vẫn có thể cần tới `O(n)` bộ nhớ tạm |

**Vướng ở đâu?** Ta sắp xếp **toàn bộ** mảng chỉ để trả lời một câu hỏi nhỏ. Đề yêu cầu `O(n)`, nên phải bỏ được bước sắp xếp.

## 5. Cách 3 — Dùng `set` để hỏi "có `k` không?" trong O(1)

Quay về Cách 1: cái chậm là `k in nums` tốn `O(n)`. Nếu cất các số vào một **`set`** (tập hợp), câu hỏi `k in s` chỉ tốn **O(1) trung bình**.

**Kiến thức nền: `set`**

- `set` là tập hợp các giá trị **không trùng**, không có thứ tự.
- Python cài `set` bằng **hash table** (bảng băm): từ giá trị tính ra "ngăn" cần xem, nên tra cứu không phải quét hết.
- Tạo `set(nums)` tốn `O(n)` thời gian và `O(n)` bộ nhớ.

In [7]:
nums = [3, 4, -1, 1, 4]
s = set(nums)
print("set:", s)            # số 4 trùng chỉ còn một
print("1 in s ->", 1 in s)
print("2 in s ->", 2 in s)
print("số phần tử:", len(s))

set: {1, 3, 4, -1}
1 in s -> True
2 in s -> False
số phần tử: 4


## 6. Nhận xét then chốt: đáp án luôn nằm trong `[1, n + 1]`

Gọi `n = len(nums)`.

Hãy tưởng tượng có `n` cái "hộp" đánh số `1, 2, ..., n`. Mỗi phần tử của `nums` bỏ được **tối đa một** hộp (nếu giá trị của nó nằm trong `1..n`).

- Nếu **tất cả** `n` hộp đều có số → mảng chứa đúng `1, 2, ..., n` → số dương nhỏ nhất bị thiếu là `n + 1`.
- Nếu có **ít nhất một** hộp trống → hộp trống nhỏ nhất chính là đáp án, và nó `<= n`.

Không có trường hợp thứ ba. Vậy **đáp án luôn thuộc `[1, n + 1]`**. (Đây chính là "nguyên lý chuồng bồ câu": `n` con bồ câu không thể lấp quá `n` chuồng.)

**Hệ quả quan trọng:**

1. Ta chỉ cần hỏi tối đa `n + 1` câu: "có 1?", ..., "có n + 1?". Vậy Cách 1 dừng sau tối đa `n + 1` lần `in` — đúng như đã nói là `O(n²)`.
2. Các số `<= 0` hoặc `> n` là **rác**: chúng không lấp hộp nào trong `1..n`, nên thay chúng bằng giá trị rác khác cũng không đổi đáp án.

Kiểm chứng bằng code:

In [8]:
tests = [[1, 2, 0], [3, 4, -1, 1], [7, 8, 9, 11, 12], [1, 2, 3], [2, 2, 2]]
for nums in tests:
    n = len(nums)
    ans = first_missing_bruteforce(nums)
    print(f"nums = {nums!s:<18} n = {n}, đáp án = {ans}, nằm trong [1, {n + 1}]? {1 <= ans <= n + 1}")

# Hệ quả 2: thay mọi số rác (<= 0 hoặc > n) bằng 0 -> đáp án không đổi
nums = [7, 1, -5, 100, 2]
n = len(nums)
cleaned = [x if 1 <= x <= n else 0 for x in nums]
print("\nGốc   :", nums, "->", first_missing_bruteforce(nums))
print("Đã lọc:", cleaned, "->", first_missing_bruteforce(cleaned))

nums = [1, 2, 0]          n = 3, đáp án = 3, nằm trong [1, 4]? True
nums = [3, 4, -1, 1]      n = 4, đáp án = 2, nằm trong [1, 5]? True
nums = [7, 8, 9, 11, 12]  n = 5, đáp án = 1, nằm trong [1, 6]? True
nums = [1, 2, 3]          n = 3, đáp án = 4, nằm trong [1, 4]? True
nums = [2, 2, 2]          n = 3, đáp án = 1, nằm trong [1, 4]? True

Gốc   : [7, 1, -5, 100, 2] -> 3
Đã lọc: [0, 1, 0, 0, 2] -> 3


In [9]:
# Code Cách 3. Nhờ nhận xét then chốt, chỉ cần thử k = 1, 2, ..., n + 1.
# range(1, n + 2) cho ra 1, 2, ..., n + 1 (đầu mút phải n + 2 KHÔNG được lấy):
print(list(range(1, 4 + 2)))   # n = 4 -> [1, 2, 3, 4, 5]


def first_missing_set(nums: List[int]) -> int:
    seen = set(nums)
    n = len(nums)
    for k in range(1, n + 2):
        if k not in seen:
            return k          # luôn return ở đây, vì đáp án <= n + 1


for nums, expected in examples:
    print(nums, "->", first_missing_set(nums), "| mong đợi:", expected)

[1, 2, 3, 4, 5]
[1, 2, 0] -> 3 | mong đợi: 3
[3, 4, -1, 1] -> 2 | mong đợi: 2
[7, 8, 9, 11, 12] -> 1 | mong đợi: 1


| | Cách 3 |
|---|---|
| Thời gian | `O(n)` trung bình (tạo set `O(n)` + tối đa `n + 1` lần tra `O(1)` trung bình) |
| Bộ nhớ phụ | `O(n)` cho `set` |

Đây là lời giải đủ tốt cho phỏng vấn thông thường. Nhưng đề **bắt buộc O(1) bộ nhớ phụ**.

**Vướng ở đâu?** Ta cần một "bảng ghi nhớ" cho `n` hộp `1..n`, mà lại không được tạo cấu trúc mới.

**Mấu chốt:** mảng `nums` có **đúng `n` ô**, và ta chỉ quan tâm `n` hộp `1..n`. Vậy hãy **dùng chính các ô của `nums` làm hộp**:

```text
giá trị 1  <->  chỉ số 0
giá trị 2  <->  chỉ số 1
...
giá trị x  <->  chỉ số x - 1
giá trị n  <->  chỉ số n - 1
```

Kỹ thuật này gọi là **index-as-hash** (dùng chỉ số làm bảng băm). Có hai cách hiện thực: đánh dấu bằng **dấu âm** (Cách 4) và **cyclic sort** (Cách 5, cách chính).

## 7. Kiến thức nền 2 — chỉ số bắt đầu từ 0, giá trị bắt đầu từ 1

List Python đánh chỉ số từ `0`, còn giá trị ta quan tâm bắt đầu từ `1`. Vì vậy "nhà" của giá trị `x` là chỉ số `x - 1`. Nhầm chỗ này là lỗi off-by-one (lệch 1) phổ biến nhất của bài.

Cẩn thận thêm: Python cho phép **chỉ số âm** (`nums[-1]` là phần tử cuối). Nếu lỡ tính ra chỉ số âm, Python **không báo lỗi** mà lặng lẽ đọc nhầm phần tử từ cuối lên. Vì thế ta luôn phải kiểm tra `1 <= x <= n` trước khi dùng `x - 1` làm chỉ số.

Một kỹ năng nữa: các hàm ở Cách 4 và Cách 5 **sửa trực tiếp** `nums`. Khi test, ta truyền **bản sao** `nums[:]` (slicing lấy toàn bộ) hoặc `list(nums)` để mảng gốc không bị đổi giữa các lần chạy. Cell dưới thử cả hai ý.

In [10]:
nums = [10, 20, 30, 40]
n = len(nums)
for x in [1, 4, 0, -1, 5]:
    if 1 <= x <= n:
        print(f"x = {x:>2}: hợp lệ, nhà ở chỉ số {x - 1}")
    else:
        print(f"x = {x:>2}: KHÔNG hợp lệ (nếu cứ dùng nums[{x - 1}] ...)", end=" ")
        try:
            print("Python đọc ra", nums[x - 1], "-> sai lặng lẽ!")
        except IndexError:
            print("-> IndexError")

# Bản sao vs. chính list
def set_first_to_zero(arr):
    arr[0] = 0          # sửa in-place

a = [5, 6, 7]
set_first_to_zero(a[:])     # truyền bản sao
print("truyền a[:] -> a =", a)
set_first_to_zero(a)        # truyền chính a
print("truyền a    -> a =", a)

x =  1: hợp lệ, nhà ở chỉ số 0
x =  4: hợp lệ, nhà ở chỉ số 3
x =  0: KHÔNG hợp lệ (nếu cứ dùng nums[-1] ...) Python đọc ra 40 -> sai lặng lẽ!
x = -1: KHÔNG hợp lệ (nếu cứ dùng nums[-2] ...) Python đọc ra 30 -> sai lặng lẽ!
x =  5: KHÔNG hợp lệ (nếu cứ dùng nums[4] ...) -> IndexError
truyền a[:] -> a = [5, 6, 7]
truyền a    -> a = [0, 6, 7]


## 8. Cách 4 — Đánh dấu bằng dấu âm

Ý tưởng: ô chỉ số `x - 1` sẽ đóng vai "cờ" cho câu hỏi "có số `x` không?". Ta **lật ô đó thành số âm** để nói "có". Cuối cùng, ô **dương** đầu tiên ở chỉ số `i` nghĩa là thiếu số `i + 1`.

Nhưng mảng đang có sẵn số âm/0 — chúng sẽ làm rối "cờ". Nên làm 3 bước:

**Bước 1 — Dọn rác.** Thay mọi số `<= 0` hoặc `> n` bằng `n + 1`. (Hệ quả 2 ở mục 6: rác thay bằng rác khác không đổi đáp án. Chọn `n + 1` vì nó **dương** và **ngoài** `1..n`.) Sau bước này mọi số đều dương.

**Bước 2 — Đánh dấu.** Với mỗi ô, đọc `v = abs(nums[i])` (ô có thể đã bị lật âm bởi bước trước, nên phải lấy trị tuyệt đối để biết giá trị gốc). Nếu `1 <= v <= n` thì đặt `nums[v - 1] = -abs(nums[v - 1])` ("chắc chắn âm", không lật ngược nếu đã âm — quan trọng khi có số trùng).

**Bước 3 — Tìm ô dương đầu tiên.** Ô `i` còn dương → không ai đánh dấu → thiếu `i + 1`. Nếu tất cả đều âm → đáp án `n + 1`.

**Kiến thức nền: `abs`.** `abs(x)` là trị tuyệt đối: bỏ dấu âm. Vậy `-abs(x)` luôn `<= 0` dù `x` âm hay dương.

In [11]:
for x in [5, -5]:
    print(f"x = {x:>2}: abs(x) = {abs(x)}, -abs(x) = {-abs(x)}, -x = {-x}")
# Chú ý: -x lật dấu, nên lật 2 lần sẽ quay về dương. -abs(x) thì luôn âm.

x =  5: abs(x) = 5, -abs(x) = -5, -x = -5
x = -5: abs(x) = 5, -abs(x) = -5, -x = 5


In [12]:
def first_missing_negative_mark(nums: List[int]) -> int:
    """Sửa trực tiếp nums. O(n) thời gian, O(1) bộ nhớ phụ."""
    n = len(nums)
    # Bước 1: dọn rác
    for i in range(n):
        if nums[i] <= 0 or nums[i] > n:
            nums[i] = n + 1
    # Bước 2: đánh dấu
    for i in range(n):
        v = abs(nums[i])
        if v <= n:                       # sau bước 1, v luôn >= 1
            nums[v - 1] = -abs(nums[v - 1])
    # Bước 3: ô dương đầu tiên
    for i in range(n):
        if nums[i] > 0:
            return i + 1
    return n + 1


for nums, expected in examples:
    print(nums, "->", first_missing_negative_mark(nums[:]), "| mong đợi:", expected)

[1, 2, 0] -> 3 | mong đợi: 3
[3, 4, -1, 1] -> 2 | mong đợi: 2
[7, 8, 9, 11, 12] -> 1 | mong đợi: 1


### Trace Cách 4 với `nums = [4, 1, -2, 1, 7]` (n = 5)

Input này có đủ: số âm, số quá lớn (`7`), số trùng (`1` hai lần).

**Bước 1:** `-2` và `7` nằm ngoài `[1, 5]` → thay bằng `6`: `[4, 1, 6, 1, 6]`.

**Bước 2:**

| i | `nums[i]` | `v = abs(nums[i])` | Hành động | `nums` sau bước |
|---:|---:|---:|---|---|
| 0 | 4 | 4 | đánh dấu chỉ số 3 | `[4, 1, 6, -1, 6]` |
| 1 | 1 | 1 | đánh dấu chỉ số 0 | `[-4, 1, 6, -1, 6]` |
| 2 | 6 | 6 | `6 > n` → bỏ qua | `[-4, 1, 6, -1, 6]` |
| 3 | -1 | 1 | chỉ số 0 đã âm, `-abs` giữ nguyên | `[-4, 1, 6, -1, 6]` |
| 4 | 6 | 6 | bỏ qua | `[-4, 1, 6, -1, 6]` |

Hàng `i = 3` cho thấy hai lý do của `abs`: ô này đã bị lật thành `-1` nên phải `abs` mới biết giá trị gốc là `1`; và số `1` trùng không được lật chỉ số 0 về dương lại.

**Bước 3:** chỉ số 0 âm (có 1), chỉ số 1 dương → **thiếu 2**.

In [13]:
def trace_negative_mark(nums):
    nums = list(nums)
    n = len(nums)
    print(f"Ban đầu: {nums} (n = {n})")
    for i in range(n):
        if nums[i] <= 0 or nums[i] > n:
            nums[i] = n + 1
    print(f"Bước 1 - thay số ngoài [1, {n}] bằng {n + 1}: {nums}")
    print("Bước 2 - đánh dấu:")
    for i in range(n):
        v = abs(nums[i])
        if v <= n:
            nums[v - 1] = -abs(nums[v - 1])
            print(f"  i = {i}: v = {v} -> chỉ số {v - 1} thành âm: {nums}")
        else:
            print(f"  i = {i}: v = {v} > n -> bỏ qua")
    print("Bước 3 - tìm ô dương đầu tiên:")
    for i in range(n):
        if nums[i] > 0:
            print(f"  chỉ số {i} dương -> thiếu số {i + 1}")
            return i + 1
    print(f"  tất cả âm -> đáp án {n + 1}")
    return n + 1

trace_negative_mark([4, 1, -2, 1, 7])

Ban đầu: [4, 1, -2, 1, 7] (n = 5)
Bước 1 - thay số ngoài [1, 5] bằng 6: [4, 1, 6, 1, 6]
Bước 2 - đánh dấu:
  i = 0: v = 4 -> chỉ số 3 thành âm: [4, 1, 6, -1, 6]
  i = 1: v = 1 -> chỉ số 0 thành âm: [-4, 1, 6, -1, 6]
  i = 2: v = 6 > n -> bỏ qua
  i = 3: v = 1 -> chỉ số 0 thành âm: [-4, 1, 6, -1, 6]
  i = 4: v = 6 > n -> bỏ qua
Bước 3 - tìm ô dương đầu tiên:
  chỉ số 1 dương -> thiếu số 2


2

| | Cách 4 |
|---|---|
| Thời gian | `O(n)`: ba vòng `for` độc lập, mỗi vòng `n` bước |
| Bộ nhớ phụ | `O(1)`: chỉ dùng `n`, `i`, `v` |

Cách này đã **đạt yêu cầu đề**. Điểm trừ nhỏ: phải "mượn" dấu âm làm cờ và nhớ dùng `abs` đúng chỗ. Cách 5 dưới đây trực quan hơn: thay vì cắm cờ, ta **dọn từng số về đúng nhà**.

## 9. Cách 5 (cách chính) — Cyclic sort: đưa mỗi số `x` về nhà ở chỉ số `x - 1`

Hình dung: mảng là một dãy `n` căn nhà, nhà số `i` (chỉ số) dành cho người mang số `i + 1`. Ta muốn sắp xếp sao cho:

```text
nums[0] = 1, nums[1] = 2, ..., nums[n-1] = n     (nếu các số đó có mặt)
```

Sau khi sắp xong, ta đi từ trái sang: căn nhà `i` đầu tiên có `nums[i] != i + 1` → **thiếu `i + 1`**. Nếu mọi nhà đều đúng → đáp án `n + 1`.

**Cách sắp:** đứng ở vị trí `i`, nhìn `x = nums[i]`:

- Nếu `x` không thuộc `[1, n]` → rác, không có nhà → để yên, sang `i + 1`.
- Nếu nhà của `x` (chỉ số `j = x - 1`) **đã có** số `x` → để yên, sang `i + 1`. (Trường hợp này gồm cả `j == i` — `x` đã ở đúng nhà — và trường hợp **số trùng**: nhà đã có một bản `x` khác.)
- Ngược lại → **swap** `nums[i]` với `nums[j]`. Bây giờ `x` đã về đúng nhà. Nhưng ở vị trí `i` lại có một số **mới** chuyển tới, nên **chưa** được sang `i + 1`: lặp lại kiểm tra tại `i`.

Vì "lặp lại tại `i`" nên ta dùng `while` bên trong `for`. Trông như `O(n²)`, nhưng mục 11 sẽ chứng minh tổng cộng chỉ có tối đa `n` lần swap.

Trước khi code, cần học kỹ thao tác swap.

### Kiến thức nền 3 — swap trong Python và cái bẫy chết người

Python swap hai biến rất gọn:

```python
a, b = b, a
```

Python làm **2 pha**: (1) tính hết vế phải thành một tuple `(b, a)`, (2) gán **lần lượt từ trái sang phải** vào các đích ở vế trái.

Với hai chỉ số đã tính sẵn `i`, `j`, swap trong list an toàn:

```python
nums[i], nums[j] = nums[j], nums[i]
```

Nhưng nếu viết thẳng `nums[nums[i] - 1]` ở vế trái, chỉ số của đích thứ hai được tính **sau khi** `nums[i]` đã bị gán giá trị mới → swap sai!

In [14]:
a, b = 1, 2
a, b = b, a
print("swap biến:", a, b)

# Cách an toàn: tính j trước
nums = [2, 1]
i = 0
j = nums[i] - 1
nums[i], nums[j] = nums[j], nums[i]
print("tính j trước     :", nums)        # [1, 2] đúng

# Cái bẫy: viết thẳng nums[nums[i] - 1] ở vế trái
nums = [2, 1]
i = 0
nums[i], nums[nums[i] - 1] = nums[nums[i] - 1], nums[i]
print("viết thẳng một dòng:", nums)      # vẫn [2, 1] -> không swap được!

swap biến: 2 1
tính j trước     : [1, 2]
viết thẳng một dòng: [2, 1]


Vì sao bản "một dòng" hỏng với `nums = [2, 1]`, `i = 0`?

1. Vế phải: `nums[nums[0] - 1] = nums[1] = 1` và `nums[0] = 2` → tuple `(1, 2)`.
2. Gán đích thứ nhất: `nums[0] = 1` → `nums = [1, 1]`.
3. Gán đích thứ hai: lúc này `nums[0]` đã là `1`, nên `nums[nums[0] - 1]` là `nums[0]`! Gán `nums[0] = 2` → `nums = [2, 1]`.

Mảng trở lại như cũ. Trong vòng `while` của cyclic sort, việc này gây **lặp vô hạn**. **Quy tắc: luôn tính `j = nums[i] - 1` ra biến riêng trước khi swap.**

In [15]:
def first_missing_cyclic_sort(nums: List[int]) -> int:
    """Sửa trực tiếp nums. O(n) thời gian, O(1) bộ nhớ phụ."""
    n = len(nums)
    # Pha 1: đưa mỗi số x thuộc [1, n] về chỉ số x - 1
    for i in range(n):
        while 1 <= nums[i] <= n and nums[nums[i] - 1] != nums[i]:
            j = nums[i] - 1
            nums[i], nums[j] = nums[j], nums[i]
    # Pha 2: tìm nhà đầu tiên sai chủ
    for i in range(n):
        if nums[i] != i + 1:
            return i + 1
    return n + 1


for nums, expected in examples:
    print(nums, "->", first_missing_cyclic_sort(nums[:]), "| mong đợi:", expected)

[1, 2, 0] -> 3 | mong đợi: 3
[3, 4, -1, 1] -> 2 | mong đợi: 2
[7, 8, 9, 11, 12] -> 1 | mong đợi: 1


### Đọc kỹ điều kiện `while`

```python
while 1 <= nums[i] <= n and nums[nums[i] - 1] != nums[i]:
```

- `1 <= nums[i] <= n`: chỉ số có nhà mới được di chuyển. Python cho viết so sánh kép `1 <= x <= n`, nghĩa là `1 <= x and x <= n`. Vế này đứng **trước** `and` nên nếu sai, vế sau (có `nums[nums[i] - 1]`) không bao giờ được tính → không đọc chỉ số âm/vượt biên (gọi là **short-circuit**).
- `nums[nums[i] - 1] != nums[i]`: nhà của `nums[i]` chưa có đúng số đó. Nếu nhà đã có rồi (kể cả khi chính là `i`), swap là vô ích → dừng. Đây là điều kiện **chống lặp vô hạn khi có số trùng**.

Pha 2 rất giống Bước 3 của Cách 4: nhà đầu tiên "sai chủ" cho ra đáp án.

## 10. Trace Cách 5 từng bước

### Ví dụ A: `nums = [3, 4, -1, 1]` (n = 4, đáp án 2)

| Bước | i | `nums[i]` | nhà `j` | `nums[j]` | Quyết định | `nums` sau bước |
|---:|---:|---:|---:|---:|---|---|
| 1 | 0 | 3 | 2 | -1 | `-1 != 3` → swap | `[-1, 4, 3, 1]` |
| 2 | 0 | -1 | — | — | ngoài `[1, 4]` → sang `i = 1` | `[-1, 4, 3, 1]` |
| 3 | 1 | 4 | 3 | 1 | swap | `[-1, 1, 3, 4]` |
| 4 | 1 | 1 | 0 | -1 | swap (số mới tới ô 1 vẫn cần về nhà) | `[1, -1, 3, 4]` |
| 5 | 1 | -1 | — | — | ngoài `[1, 4]` → sang `i = 2` | `[1, -1, 3, 4]` |
| 6 | 2 | 3 | 2 | 3 | đã đúng nhà → sang `i = 3` | `[1, -1, 3, 4]` |
| 7 | 3 | 4 | 3 | 4 | đã đúng nhà → hết pha 1 | `[1, -1, 3, 4]` |

Pha 2: chỉ số 0 chứa `1` (đúng), chỉ số 1 chứa `-1` (phải là `2`) → **đáp án 2**. Tổng cộng 3 lần swap.

Chú ý bước 3–4: ở `i = 1`, `while` chạy **2 lần swap liên tiếp** trước khi sang `i = 2`. Đây là lý do cần `while` chứ không phải `if`.

In [16]:
def trace_cyclic_sort(nums):
    nums = list(nums)
    n = len(nums)
    swaps = 0
    print(f"Ban đầu: {nums} (n = {n})")
    for i in range(n):
        while True:
            x = nums[i]
            if not (1 <= x <= n):
                print(f"  i = {i}: nums[{i}] = {x} ngoài [1, {n}] -> sang i kế")
                break
            j = x - 1
            if nums[j] == x:
                why = "đã đúng nhà" if j == i else f"nhà {j} đã có {x} (số trùng)"
                print(f"  i = {i}: nums[{i}] = {x}, {why} -> sang i kế")
                break
            nums[i], nums[j] = nums[j], nums[i]
            swaps += 1
            print(f"  i = {i}: swap đưa {x} về chỉ số {j} -> {nums}")
    print(f"Sau pha 1: {nums}, số swap = {swaps}")
    for i in range(n):
        if nums[i] != i + 1:
            print(f"Pha 2: chỉ số {i} chứa {nums[i]} != {i + 1} -> đáp án {i + 1}")
            return i + 1
    print(f"Pha 2: mọi nhà đều đúng -> đáp án {n + 1}")
    return n + 1

trace_cyclic_sort([3, 4, -1, 1])

Ban đầu: [3, 4, -1, 1] (n = 4)
  i = 0: swap đưa 3 về chỉ số 2 -> [-1, 4, 3, 1]
  i = 0: nums[0] = -1 ngoài [1, 4] -> sang i kế
  i = 1: swap đưa 4 về chỉ số 3 -> [-1, 1, 3, 4]
  i = 1: swap đưa 1 về chỉ số 0 -> [1, -1, 3, 4]
  i = 1: nums[1] = -1 ngoài [1, 4] -> sang i kế
  i = 2: nums[2] = 3, đã đúng nhà -> sang i kế
  i = 3: nums[3] = 4, đã đúng nhà -> sang i kế
Sau pha 1: [1, -1, 3, 4], số swap = 3
Pha 2: chỉ số 1 chứa -1 != 2 -> đáp án 2


2

### Ví dụ B: có số trùng — `nums = [2, 2, 5, 1, 0]` (n = 5)

Dự đoán trước khi chạy:

- `i = 0`: `nums[0] = 2`, nhà là chỉ số 1, mà `nums[1]` cũng là `2` → **không swap** (nếu swap thì hai số `2` đổi chỗ cho nhau mãi mãi).
- `i = 3`: `1` được đưa về chỉ số 0, kéo `2` ra chỗ chỉ số 3; số `2` này lại thấy nhà đã có `2` → dừng.
- Sau pha 1, chỉ số 2 chứa `0` → thiếu `3`.

In [17]:
trace_cyclic_sort([2, 2, 5, 1, 0])

Ban đầu: [2, 2, 5, 1, 0] (n = 5)
  i = 0: nums[0] = 2, nhà 1 đã có 2 (số trùng) -> sang i kế
  i = 1: nums[1] = 2, đã đúng nhà -> sang i kế
  i = 2: swap đưa 5 về chỉ số 4 -> [2, 2, 0, 1, 5]
  i = 2: nums[2] = 0 ngoài [1, 5] -> sang i kế
  i = 3: swap đưa 1 về chỉ số 0 -> [1, 2, 0, 2, 5]
  i = 3: nums[3] = 2, nhà 1 đã có 2 (số trùng) -> sang i kế
  i = 4: nums[4] = 5, đã đúng nhà -> sang i kế
Sau pha 1: [1, 2, 0, 2, 5], số swap = 2
Pha 2: chỉ số 2 chứa 0 != 3 -> đáp án 3


3

## 11. Vì sao tổng số swap ≤ n (nên Cách 5 là O(n))?

Có `while` lồng trong `for` nhưng đừng vội kết luận `O(n²)`. Hãy đếm **tổng** số lần swap trên toàn bộ thuật toán, thay vì đếm theo từng `i`.

Gọi một chỉ số `k` là **"đã ổn định"** nếu `nums[k] == k + 1`.

**Ý 1 — Mỗi swap tạo ra ít nhất một chỉ số ổn định mới.** Swap chỉ xảy ra khi `nums[j] != x` (với `j = x - 1`), tức chỉ số `j` **chưa** ổn định. Sau swap, `nums[j] = x = j + 1` → chỉ số `j` **trở thành** ổn định.

**Ý 2 — Chỉ số đã ổn định không bao giờ bị đụng lại.** Giả sử `nums[k] == k + 1`.
- `k` không thể là đích `j` của một swap: muốn `j == k` thì `nums[i] = k + 1 = nums[k]`, nhưng khi đó điều kiện `nums[j] != nums[i]` sai → không swap.
- `k` không thể là nguồn `i` của một swap: khi `i == k`, ta có `j = nums[k] - 1 = k` và `nums[j] == nums[i]` → không swap.

**Kết luận:** mỗi swap "khóa" vĩnh viễn thêm ít nhất một chỉ số trong `n` chỉ số → **tối đa `n` swap** cho cả thuật toán.

Tổng số lần kiểm tra điều kiện `while` = (số lần điều kiện sai, đúng 1 lần cho mỗi `i`, tức `n`) + (số lần đúng, tức số swap, `<= n`) `<= 2n`. Cộng thêm pha 2 là `n` → **O(n)**.

Kỹ thuật "đếm tổng trên toàn thuật toán" này gọi là **phân tích khấu hao** (amortized analysis): có `i` tốn nhiều bước, nhưng nó "dùng hết phần" của các `i` sau.

In [18]:
def cyclic_sort_count_swaps(nums):
    """Chạy pha 1, trả về số swap; đồng thời kiểm tra Ý 2 (ô ổn định không bị đụng lại)."""
    nums = list(nums)
    n = len(nums)
    swaps = 0
    for i in range(n):
        while 1 <= nums[i] <= n and nums[nums[i] - 1] != nums[i]:
            j = nums[i] - 1
            assert nums[i] != i + 1 and nums[j] != j + 1, "đụng vào ô đã ổn định!"
            nums[i], nums[j] = nums[j], nums[i]
            swaps += 1
    return swaps

# Một i có thể làm rất nhiều swap: [2, 3, 4, ..., n, 1]
for n in [5, 10, 1000]:
    worst = list(range(2, n + 1)) + [1]
    print(f"n = {n:>4}: [2, 3, ..., {n}, 1] cần {cyclic_sort_count_swaps(worst)} swap (<= n)")

# Thử ngẫu nhiên: số swap không bao giờ vượt n
random.seed(41)
max_ratio = 0
for _ in range(2000):
    n = random.randint(1, 12)
    arr = [random.randint(-3, n + 3) for _ in range(n)]
    s = cyclic_sort_count_swaps(arr)
    assert s <= n
    max_ratio = max(max_ratio, s / n)
print("2000 mảng ngẫu nhiên: luôn có swap <= n; tỉ lệ swap/n lớn nhất =", round(max_ratio, 3))

n =    5: [2, 3, ..., 5, 1] cần 4 swap (<= n)
n =   10: [2, 3, ..., 10, 1] cần 9 swap (<= n)
n = 1000: [2, 3, ..., 1000, 1] cần 999 swap (<= n)
2000 mảng ngẫu nhiên: luôn có swap <= n; tỉ lệ swap/n lớn nhất = 0.857


Ví dụ `[2, 3, 4, 5, 1]`: ngay tại `i = 0`, `while` chạy 4 swap liên tiếp (2 về nhà, rồi 3, rồi 4, rồi 5, cuối cùng 1 tới chỉ số 0). Sau đó mọi `i` còn lại đều 0 swap. Tổng `n - 1 = 4`, đúng như phân tích.

Tỉ lệ swap/n lớn nhất < 1 cũng khớp lý thuyết, thậm chí chặt hơn: số swap không vượt quá số ô ổn định lúc kết thúc. Nếu mảng thiếu ít nhất một số của `1..n` thì có tối đa `n - 1` ô ổn định; nếu đủ cả `1..n` (một hoán vị) thì lần swap cuối của mỗi "vòng đổi chỗ" ổn định **hai** ô cùng lúc. Cả hai trường hợp đều cho tối đa `n - 1` swap.

## 12. Code nộp LeetCode — Cyclic sort (cách chính)

Trên LeetCode, `List` đã được import sẵn. Trong notebook ta đã `from typing import List` ở đầu.

In [19]:
class Solution:
    def firstMissingPositive(self, nums: List[int]) -> int:
        n = len(nums)
        for i in range(n):
            while 1 <= nums[i] <= n and nums[nums[i] - 1] != nums[i]:
                j = nums[i] - 1
                nums[i], nums[j] = nums[j], nums[i]
        for i in range(n):
            if nums[i] != i + 1:
                return i + 1
        return n + 1

### Test bằng assert: ví dụ đề + edge case

| Nhóm | Input | Đáp án | Vì sao đáng test |
|---|---|---:|---|
| Đề | `[1,2,0]`, `[3,4,-1,1]`, `[7,8,9,11,12]` | 3, 2, 1 | ba kiểu kết quả: `n+1`, giữa, `1` |
| Một phần tử | `[1]`, `[2]`, `[-5]`, `[0]` | 2, 1, 1, 1 | `n = 1` |
| Đủ `1..n` | `[1,2,3]`, `[3,1,2]` | 4, 4 | đáp án `n + 1` |
| Toàn số trùng | `[1,1]`, `[2,2,2]` | 2, 1 | chống lặp vô hạn |
| Số biên | `[2147483647]`, `[-2147483648, 1]` | 1, 2 | giá trị cực lớn/nhỏ |
| Đảo ngược dài | `[100000, ..., 1]` | 100001 | `n` lớn nhất, kiểm tra tốc độ |

In [20]:
def check(name, nums, expected):
    actual = Solution().firstMissingPositive(nums[:])
    assert actual == expected, (name, nums, actual, expected)
    print(f"{name:<22} PASS (= {expected})")

check("đề ví dụ 1", [1, 2, 0], 3)
check("đề ví dụ 2", [3, 4, -1, 1], 2)
check("đề ví dụ 3", [7, 8, 9, 11, 12], 1)
check("một phần tử [1]", [1], 2)
check("một phần tử [2]", [2], 1)
check("một phần tử âm", [-5], 1)
check("một phần tử 0", [0], 1)
check("đủ 1..n", [1, 2, 3], 4)
check("đủ 1..n xáo trộn", [3, 1, 2], 4)
check("trùng [1,1]", [1, 1], 2)
check("trùng [2,2,2]", [2, 2, 2], 1)
check("số cực lớn", [2147483647], 1)
check("số cực nhỏ", [-2147483648, 1], 2)
check("n = 10^5 đảo ngược", list(range(100000, 0, -1)), 100001)
print("Tất cả test PASS!")

đề ví dụ 1             PASS (= 3)
đề ví dụ 2             PASS (= 2)
đề ví dụ 3             PASS (= 1)
một phần tử [1]        PASS (= 2)
một phần tử [2]        PASS (= 1)
một phần tử âm         PASS (= 1)
một phần tử 0          PASS (= 1)
đủ 1..n                PASS (= 4)
đủ 1..n xáo trộn       PASS (= 4)
trùng [1,1]            PASS (= 2)
trùng [2,2,2]          PASS (= 1)
số cực lớn             PASS (= 1)
số cực nhỏ             PASS (= 2)
n = 10^5 đảo ngược     PASS (= 100001)
Tất cả test PASS!


### Bản nộp thứ hai (tuỳ chọn) — Đánh dấu bằng dấu âm

Nếu bạn thấy Cách 4 dễ nhớ hơn, đây là bản nộp tương ứng. Nó cũng đạt `O(n)` / `O(1)`.

### Test đối chiếu chéo cả 5 cách trên input ngẫu nhiên

Ngay sau bản nộp thứ hai, ta cho mọi hàm "thi đấu" với brute force. Mảng nhỏ (dài 1–10), giá trị trong `[-3, 12]` để dễ có số âm, số 0, số trùng, số `> n`. Brute force là "trọng tài" vì nó đơn giản nhất, khó sai nhất. `random.seed` cố định để lần nào chạy cũng ra cùng bộ test.

In [21]:
class SolutionNegativeMark:
    def firstMissingPositive(self, nums: List[int]) -> int:
        n = len(nums)
        for i in range(n):
            if nums[i] <= 0 or nums[i] > n:
                nums[i] = n + 1
        for i in range(n):
            v = abs(nums[i])
            if v <= n:
                nums[v - 1] = -abs(nums[v - 1])
        for i in range(n):
            if nums[i] > 0:
                return i + 1
        return n + 1

for nums, expected in examples + [([1, 1], 2), ([2147483647], 1), ([1, 2, 3], 4)]:
    assert SolutionNegativeMark().firstMissingPositive(nums[:]) == expected
print("SolutionNegativeMark PASS!")

# ---- Test đối chiếu chéo ----
random.seed(2026)
methods = {
    "sort": first_missing_sort,
    "set": first_missing_set,
    "negative_mark": first_missing_negative_mark,
    "cyclic_sort": first_missing_cyclic_sort,
    "Solution": Solution().firstMissingPositive,
    "SolutionNegativeMark": SolutionNegativeMark().firstMissingPositive,
}
for _ in range(3000):
    n = random.randint(1, 10)
    nums = [random.randint(-3, 12) for _ in range(n)]
    expected = first_missing_bruteforce(nums)
    for name, f in methods.items():
        got = f(nums[:])                       # bản sao: 2 cách sửa in-place
        assert got == expected, (name, nums, got, expected)
print("3000 mảng ngẫu nhiên: 6 hàm đều khớp brute force!")

SolutionNegativeMark PASS!
3000 mảng ngẫu nhiên: 6 hàm đều khớp brute force!


## 13. Vì sao đúng? (invariant bằng lời)

**Nền tảng chung (mục 6):** đáp án thuộc `[1, n + 1]`, và chỉ các số trong `[1, n]` mới ảnh hưởng tới đáp án. Hai cách O(1) đều chỉ cần trả lời đúng câu hỏi "số `k` có mặt không?" cho mọi `k` trong `1..n`.

**Cách 4 (dấu âm):**
- Sau bước 1, mọi ô dương, và tập các số thuộc `[1, n]` có mặt **không đổi** (chỉ rác bị thay bằng rác).
- Invariant của bước 2: `abs(nums[i])` luôn bằng giá trị sau bước 1 của ô `i` (ta chỉ đổi dấu, không đổi độ lớn). Nhờ vậy đọc `abs` vẫn ra đúng giá trị cần xử lý.
- Sau bước 2: ô `k - 1` âm **khi và chỉ khi** có số `k` trong mảng (`-abs` đảm bảo đã âm thì giữ âm; không ai làm ô nào dương lại).
- Bước 3 trả về `k` nhỏ nhất có ô `k - 1` dương = số nhỏ nhất bị thiếu.

**Cách 5 (cyclic sort):**
- Swap chỉ đổi chỗ, nên **tập giá trị** (kể cả số lần lặp) trong mảng không đổi.
- Sau pha 1, với mọi `i`: hoặc `nums[i] == i + 1`, hoặc `nums[i]` là rác / bản trùng. (Khi vòng `while` ở `i` dừng, `nums[i]` hoặc ngoài `[1, n]`, hoặc nhà của nó đã có đúng nó; và ô ổn định không bao giờ bị đụng lại.)
- Vì vậy: nếu số `k` thuộc `[1, n]` có mặt thì nó **chắc chắn** nằm ở chỉ số `k - 1`. Ngược lại nếu `nums[k - 1] != k` thì `k` vắng mặt.
- Pha 2 trả về `k` nhỏ nhất có `nums[k - 1] != k` → đúng là số nhỏ nhất bị thiếu; nếu không có thì đủ `1..n` → `n + 1`.

## 14. Bảng so sánh các cách

Gọi `n = len(nums)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Sửa `nums`? | Đạt đề? |
|---|---|---:|---:|:---:|:---:|
| 1. Brute force | thử `k = 1, 2, ...` với `k in nums` | `O(n²)` | `O(1)` | không | không (TLE) |
| 2. Sort + quét | sắp xếp, đi tìm lỗ hổng | `O(n log n)` | `O(n)` (`sorted`) | không | không |
| 3. Set | `k in set` cho `k` trong `1..n+1` | `O(n)` TB | `O(n)` | không | không (bộ nhớ) |
| 4. Dấu âm | ô `x-1` âm ⇔ có `x` | `O(n)` | `O(1)` | **có** | **có** |
| 5. Cyclic sort | đưa `x` về chỉ số `x-1` | `O(n)` (≤ n swap) | `O(1)` | **có** | **có** |

Khi nào dùng gì?

- Phỏng vấn mà không bắt O(1) bộ nhớ → Cách 3 là đủ, ngắn và khó sai.
- Đề bắt O(1) → Cách 5 (dễ hình dung "về nhà") hoặc Cách 4 (không có `while` lồng). Hãy nói rõ với người phỏng vấn rằng hai cách này **làm hỏng mảng input**.

## 15. Lỗi thường gặp

### Lỗi 1: Swap một dòng làm hỏng chỉ số (đã thấy ở mục 9)

```python
# SAI: đích thứ hai được tính sau khi nums[i] đã đổi
nums[i], nums[nums[i] - 1] = nums[nums[i] - 1], nums[i]
```

Sửa: `j = nums[i] - 1` rồi `nums[i], nums[j] = nums[j], nums[i]`.

### Lỗi 2: Điều kiện `while` thiếu kiểm tra số trùng → lặp vô hạn

```python
# SAI
while 1 <= nums[i] <= n and nums[i] != i + 1:
    ...swap...
```

Với `[1, 1]`: ở `i = 1`, `nums[1] = 1 != 2` nên swap với chỉ số 0, mà chỉ số 0 cũng là `1` → mảng không đổi → lặp mãi. Sửa: so sánh với **nhà đích** `nums[nums[i] - 1] != nums[i]`.

Cell dưới mô phỏng lỗi này, có "phanh" giới hạn số bước để notebook không treo:

In [22]:
def cyclic_sort_wrong_condition(nums, max_steps=20):
    nums = list(nums)
    n = len(nums)
    steps = 0
    for i in range(n):
        while 1 <= nums[i] <= n and nums[i] != i + 1:      # thiếu kiểm tra nhà đích
            j = nums[i] - 1
            nums[i], nums[j] = nums[j], nums[i]
            steps += 1
            if steps >= max_steps:
                return f"KẸT sau {steps} swap tại i = {i}, nums = {nums}"
    return f"xong, nums = {nums}"

print("[1, 1]   :", cyclic_sort_wrong_condition([1, 1]))
print("[3, 4, -1, 1]:", cyclic_sort_wrong_condition([3, 4, -1, 1]))   # không trùng thì vẫn chạy

[1, 1]   : KẸT sau 20 swap tại i = 1, nums = [1, 1]
[3, 4, -1, 1]: xong, nums = [1, -1, 3, 4]


Lỗi này nguy hiểm vì **ví dụ của đề đều qua**, chỉ tới khi gặp số trùng mới treo (LeetCode báo TLE).

### Lỗi 3: Cách 4 dùng `*= -1` thay vì `-abs(...)` → số trùng lật dấu hai lần

In [23]:
def negative_mark_wrong_flip(nums):
    n = len(nums)
    for i in range(n):
        if nums[i] <= 0 or nums[i] > n:
            nums[i] = n + 1
    for i in range(n):
        v = abs(nums[i])
        if v <= n:
            nums[v - 1] *= -1          # SAI: lật dấu, gặp lần 2 sẽ dương lại
    for i in range(n):
        if nums[i] > 0:
            return i + 1
    return n + 1

print("[1, 1] sai ->", negative_mark_wrong_flip([1, 1]), "| đúng ->", first_missing_negative_mark([1, 1]))

[1, 1] sai -> 1 | đúng -> 2


Với `[1, 1]`: số `1` đầu lật chỉ số 0 thành âm, số `1` thứ hai lật lại thành dương → bước 3 tưởng thiếu `1`. Sửa: `nums[v - 1] = -abs(nums[v - 1])`.

### Lỗi 4: Quên `abs` khi đọc giá trị ở bước 2

Nếu viết `v = nums[i]`, có lúc `v` âm (ô đã bị đánh dấu trước đó) → `nums[v - 1]` là **chỉ số âm**, Python không báo lỗi mà đọc từ cuối mảng (xem mục 7) → kết quả sai lặng lẽ.

### Lỗi 5: Cách 4 dọn rác bằng `0` thay vì `n + 1`

`0` đổi dấu vẫn là `0` (`-abs(0) == 0`), nên ô chứa `0` không thể "đánh dấu âm" được → nếu số tương ứng có mặt, ta vẫn tưởng nó thiếu. Phải dùng một số **dương** ngoài `[1, n]`, ví dụ `n + 1`.

### Lỗi 6: Quên trường hợp đủ `1..n`

```python
for i in range(n):
    if nums[i] != i + 1:
        return i + 1
# SAI nếu quên dòng dưới: hàm trả None với [1, 2, 3]
return n + 1
```

Tương tự ở Cách 3: phải là `range(1, n + 2)` chứ không phải `range(1, n + 1)`.

In [24]:
def set_wrong_range(nums):
    seen = set(nums)
    for k in range(1, len(nums) + 1):      # SAI: thiếu n + 1
        if k not in seen:
            return k

print("[1, 2, 3] sai ->", set_wrong_range([1, 2, 3]), "| đúng ->", first_missing_set([1, 2, 3]))

[1, 2, 3] sai -> None | đúng -> 4


## 16. Mẹo nhận dạng pattern

Khi thấy các dấu hiệu sau, hãy nghĩ tới **index-as-hash / cyclic sort**:

1. Mảng dài `n`, và các giá trị "đáng quan tâm" nằm trong `[1, n]` (hoặc `[0, n - 1]`, `[0, n]`).
2. Câu hỏi về **thiếu / trùng / có mặt** của các giá trị đó.
3. Đề đòi **O(1) bộ nhớ phụ** nhưng cho phép sửa mảng.

Tư duy chung: "mảng có `n` ô, ta cần ghi nhớ `n` câu trả lời có/không → mượn chính các ô làm chỗ ghi". Muốn ghi mà không mất dữ liệu cũ thì:

- dùng **dấu** (âm/dương) của ô làm cờ (Cách 4), hoặc
- **đổi chỗ** để vị trí tự nói lên giá trị (Cách 5).

Bước "giảm miền": luôn tự hỏi **đáp án có thể nằm trong khoảng nào?** Ở bài này, câu trả lời `[1, n + 1]` đã biến một bài trông như vô hạn (số tới 2 tỷ) thành bài chỉ cần `n` hộp.

## 17. Biến thể và bài liên quan

| Bài | Nội dung | Kỹ thuật |
|---|---|---|
| 268. Missing Number | `n` số khác nhau trong `[0, n]`, tìm số thiếu | tổng `0..n` trừ tổng mảng, hoặc XOR |
| 448. Find All Numbers Disappeared in an Array | giá trị trong `[1, n]`, tìm **mọi** số thiếu | dấu âm / cyclic sort (Bài tập 5) |
| 442. Find All Duplicates in an Array | mỗi số xuất hiện 1 hoặc 2 lần, tìm số xuất hiện 2 lần | cyclic sort / dấu âm (Bài tập 6) |
| 287. Find the Duplicate Number | `n + 1` số trong `[1, n]`, **không** được sửa mảng | Floyd tìm chu trình (kỹ thuật khác) |
| 645. Set Mismatch | một số bị lặp, một số bị thiếu | cyclic sort: tìm ô sai chủ |

Câu hỏi mở rộng hay gặp khi phỏng vấn: *"Nếu không được sửa `nums` thì sao?"* → khi đó không còn chỗ để ghi nhớ, ta quay về Cách 3 (`set`, `O(n)` bộ nhớ) hoặc Cách 2.

## 18. Bài tập tự luyện

Làm theo thứ tự, mỗi bài khó hơn bài trước. Đừng xem phần đáp án ở cuối notebook trước khi tự làm.

### Bài 1 — Đoán output (không chạy code)

Cho biết `firstMissingPositive` trả về gì:

```text
a) [1, 2, 3]
b) [2, 3, 4]
c) [1, 1, 1, 1]
d) [-1, -2]
e) [5, 1, 4, 2]
f) Với mảng có n = 6 phần tử bất kỳ, đáp án lớn nhất có thể là bao nhiêu? Cho một mảng đạt được giá trị đó.
```

### Bài 2 — Tính bằng tay cyclic sort

Chạy tay pha 1 của Cách 5 trên `nums = [4, 3, 4, 1, 2]` (n = 5). Lập bảng như mục 10 và trả lời:

1. Mảng sau pha 1 là gì?
2. Có tổng cộng bao nhiêu lần swap?
3. Đáp án là bao nhiêu?

(Gợi ý kiểm tra: số swap phải `<= 5`.)

### Bài 3 — Điền TODO: cyclic sort

Hoàn thiện hàm theo các TODO. Đừng quên bẫy swap một dòng.

In [25]:
def exercise_cyclic_sort(nums: List[int]) -> int:
    n = len(nums)
    for i in range(n):
        # TODO 1: viết điều kiện while: nums[i] có nhà trong [1, n] VÀ nhà đó chưa có đúng nums[i]
        # TODO 2: tính j = nhà của nums[i]
        # TODO 3: swap nums[i] và nums[j]
        pass
    # TODO 4: tìm chỉ số i đầu tiên có nums[i] != i + 1 và trả về i + 1
    # TODO 5: nếu không có, trả về n + 1
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert exercise_cyclic_sort([1, 2, 0]) == 3
# assert exercise_cyclic_sort([3, 4, -1, 1]) == 2
# assert exercise_cyclic_sort([7, 8, 9, 11, 12]) == 1
# assert exercise_cyclic_sort([1, 1]) == 2

### Bài 4 — Tự viết lại Cách 4 (đánh dấu bằng dấu âm)

Không nhìn code phía trên, tự viết đủ 3 bước: dọn rác, đánh dấu, tìm ô dương đầu tiên. Yêu cầu O(n) thời gian, O(1) bộ nhớ phụ.

In [26]:
def exercise_negative_mark(nums: List[int]) -> int:
    # TODO: bước 1 - thay số <= 0 hoặc > n bằng n + 1
    # TODO: bước 2 - với v = abs(nums[i]) <= n, đặt nums[v - 1] = -abs(nums[v - 1])
    # TODO: bước 3 - ô dương đầu tiên ở chỉ số i -> trả về i + 1; không có -> n + 1
    pass

# assert exercise_negative_mark([4, 1, -2, 1, 7]) == 2
# assert exercise_negative_mark([1, 1]) == 2
# assert exercise_negative_mark([1, 2, 3]) == 4

### Bài 5 — Biến thể: LeetCode 448 "Find All Numbers Disappeared in an Array"

Cho mảng `nums` dài `n`, **mọi** giá trị nằm trong `[1, n]` (có thể trùng). Trả về **danh sách tất cả** các số trong `[1, n]` không có mặt, theo thứ tự tăng dần. Yêu cầu O(n) thời gian, không dùng thêm list/set phụ (list kết quả không tính).

```text
[4, 3, 2, 7, 8, 2, 3, 1]  ->  [5, 6]
[1, 1]                    ->  [2]
```

In [27]:
def exercise_find_disappeared(nums: List[int]) -> List[int]:
    # TODO: dùng cyclic sort hoặc dấu âm, rồi gom MỌI chỉ số "sai chủ" / còn dương
    pass

# assert exercise_find_disappeared([4, 3, 2, 7, 8, 2, 3, 1]) == [5, 6]
# assert exercise_find_disappeared([1, 1]) == [2]
# assert exercise_find_disappeared([1, 2, 3]) == []

### Bài 6 — Mini challenge: LeetCode 442 "Find All Duplicates in an Array"

Cho mảng `nums` dài `n`, mọi giá trị trong `[1, n]`, mỗi số xuất hiện **một hoặc hai lần**. Trả về các số xuất hiện **hai lần** (ở đây yêu cầu thêm: sắp tăng dần để dễ test). O(n) thời gian, O(1) bộ nhớ phụ (không tính list kết quả).

```text
[4, 3, 2, 7, 8, 2, 3, 1]  ->  [2, 3]
[1, 1, 2]                 ->  [1]
[1]                       ->  []
```

Câu hỏi suy nghĩ thêm: sau cyclic sort, một ô "sai chủ" chứa gì?

In [28]:
def exercise_find_duplicates(nums: List[int]) -> List[int]:
    # TODO: cyclic sort, rồi quan sát các ô có nums[i] != i + 1
    pass

# assert exercise_find_duplicates([4, 3, 2, 7, 8, 2, 3, 1]) == [2, 3]
# assert exercise_find_duplicates([1, 1, 2]) == [1]
# assert exercise_find_duplicates([1]) == []

## 19. Gợi ý theo mức

**Bài 1**
- Mức 1: dùng nhận xét "đáp án thuộc `[1, n + 1]`".
- Mức 2: liệt kê các số dương có mặt, tìm "lỗ" đầu tiên.
- Mức 3: (f) đáp án lớn nhất là `n + 1` khi mảng chứa đủ `1..n`.

**Bài 2**
- Mức 1: nhớ rằng sau mỗi swap, **không** tăng `i`.
- Mức 2: dừng khi `nums[i]` ngoài `[1, n]` **hoặc** nhà của nó đã có đúng nó (kể cả số trùng).
- Mức 3: ở `i = 1` sẽ có một lần swap rồi gặp số `4` trùng; ở `i = 4` cũng swap một lần rồi gặp `4` trùng.

**Bài 3**
- Mức 1: điều kiện `while` có hai vế nối bằng `and`; vế kiểm tra miền phải đứng trước.
- Mức 2: `while 1 <= nums[i] <= n and nums[nums[i] - 1] != nums[i]:`
- Mức 3: trong thân `while`: `j = nums[i] - 1` rồi `nums[i], nums[j] = nums[j], nums[i]`. Nhớ xoá `pass` và `raise NotImplementedError`.

**Bài 4**
- Mức 1: tại sao rác phải thay bằng số **dương**? (xem Lỗi 5)
- Mức 2: luôn đọc `abs(nums[i])` vì ô có thể đã bị đánh dấu.
- Mức 3: `nums[v - 1] = -abs(nums[v - 1])`, không phải `*= -1`.

**Bài 5**
- Mức 1: giống bài chính, nhưng thay vì `return` ở ô sai đầu tiên thì **gom tất cả**.
- Mức 2: giá trị đều trong `[1, n]` nên không cần bước dọn rác.
- Mức 3: dấu âm: đánh dấu xong, `[i + 1 for i in range(n) if nums[i] > 0]`.

**Bài 6**
- Mức 1: số trùng không được swap về nhà vì nhà đã có chủ → nó "kẹt" ở ô khác.
- Mức 2: sau cyclic sort, ô `i` có `nums[i] != i + 1` chứa đúng **một bản thừa** của một số bị lặp.
- Mức 3: gom `nums[i]` ở các ô sai chủ, rồi `sorted(...)`. Mỗi số lặp đúng 2 lần nên chỉ có đúng 1 bản thừa → không bị gom trùng.

## 20. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

```text
a) [1, 2, 3]     -> 4   (đủ 1..3, n = 3 -> n + 1)
b) [2, 3, 4]     -> 1
c) [1, 1, 1, 1]  -> 2
d) [-1, -2]      -> 1
e) [5, 1, 4, 2]  -> 3
f) n + 1 = 7, ví dụ [1, 2, 3, 4, 5, 6] hoặc bất kỳ hoán vị nào của nó
```

In [29]:
sol = Solution()
answers_b1 = [([1, 2, 3], 4), ([2, 3, 4], 1), ([1, 1, 1, 1], 2), ([-1, -2], 1), ([5, 1, 4, 2], 3),
              ([1, 2, 3, 4, 5, 6], 7), ([6, 5, 4, 3, 2, 1], 7)]
for nums, expected in answers_b1:
    assert sol.firstMissingPositive(nums[:]) == expected, nums
print("Bài 1 PASS")

Bài 1 PASS


### Đáp án Bài 2

| Bước | i | `nums[i]` | nhà `j` | `nums[j]` | Quyết định | `nums` sau bước |
|---:|---:|---:|---:|---:|---|---|
| 1 | 0 | 4 | 3 | 1 | swap | `[1, 3, 4, 4, 2]` |
| 2 | 0 | 1 | 0 | 1 | đã đúng nhà | |
| 3 | 1 | 3 | 2 | 4 | swap | `[1, 4, 3, 4, 2]` |
| 4 | 1 | 4 | 3 | 4 | nhà 3 đã có 4 (trùng) | |
| 5 | 2 | 3 | 2 | 3 | đã đúng nhà | |
| 6 | 3 | 4 | 3 | 4 | đã đúng nhà | |
| 7 | 4 | 2 | 1 | 4 | swap | `[1, 2, 3, 4, 4]` |
| 8 | 4 | 4 | 3 | 4 | nhà 3 đã có 4 (trùng) | |

1. Sau pha 1: `[1, 2, 3, 4, 4]`.
2. 3 lần swap.
3. Chỉ số 4 chứa `4 != 5` → đáp án **5**.

Đối chiếu bằng `trace_cyclic_sort`:

In [30]:
assert trace_cyclic_sort([4, 3, 4, 1, 2]) == 5
assert cyclic_sort_count_swaps([4, 3, 4, 1, 2]) == 3
print("Bài 2 PASS")

Ban đầu: [4, 3, 4, 1, 2] (n = 5)
  i = 0: swap đưa 4 về chỉ số 3 -> [1, 3, 4, 4, 2]
  i = 0: nums[0] = 1, đã đúng nhà -> sang i kế
  i = 1: swap đưa 3 về chỉ số 2 -> [1, 4, 3, 4, 2]
  i = 1: nums[1] = 4, nhà 3 đã có 4 (số trùng) -> sang i kế
  i = 2: nums[2] = 3, đã đúng nhà -> sang i kế
  i = 3: nums[3] = 4, đã đúng nhà -> sang i kế
  i = 4: swap đưa 2 về chỉ số 1 -> [1, 2, 3, 4, 4]
  i = 4: nums[4] = 4, nhà 3 đã có 4 (số trùng) -> sang i kế
Sau pha 1: [1, 2, 3, 4, 4], số swap = 3
Pha 2: chỉ số 4 chứa 4 != 5 -> đáp án 5
Bài 2 PASS


In [31]:
# ===== Đáp án Bài 3, 4, 5, 6 =====
def answer_cyclic_sort(nums: List[int]) -> int:
    n = len(nums)
    for i in range(n):
        while 1 <= nums[i] <= n and nums[nums[i] - 1] != nums[i]:
            j = nums[i] - 1
            nums[i], nums[j] = nums[j], nums[i]
    for i in range(n):
        if nums[i] != i + 1:
            return i + 1
    return n + 1


def answer_negative_mark(nums: List[int]) -> int:
    n = len(nums)
    for i in range(n):
        if nums[i] <= 0 or nums[i] > n:
            nums[i] = n + 1
    for i in range(n):
        v = abs(nums[i])
        if v <= n:
            nums[v - 1] = -abs(nums[v - 1])
    for i in range(n):
        if nums[i] > 0:
            return i + 1
    return n + 1


def answer_find_disappeared(nums: List[int]) -> List[int]:
    # Dấu âm: mọi giá trị đã nằm trong [1, n] nên không cần dọn rác
    for i in range(len(nums)):
        v = abs(nums[i])
        nums[v - 1] = -abs(nums[v - 1])
    return [i + 1 for i in range(len(nums)) if nums[i] > 0]


def answer_find_duplicates(nums: List[int]) -> List[int]:
    # Cyclic sort: bản thừa của số bị lặp sẽ kẹt ở ô "sai chủ"
    n = len(nums)
    for i in range(n):
        while nums[nums[i] - 1] != nums[i]:     # mọi giá trị đã trong [1, n]
            j = nums[i] - 1
            nums[i], nums[j] = nums[j], nums[i]
    return sorted(nums[i] for i in range(n) if nums[i] != i + 1)


# Bài 3
assert answer_cyclic_sort([1, 2, 0]) == 3
assert answer_cyclic_sort([3, 4, -1, 1]) == 2
assert answer_cyclic_sort([7, 8, 9, 11, 12]) == 1
assert answer_cyclic_sort([1, 1]) == 2
# Bài 4
assert answer_negative_mark([4, 1, -2, 1, 7]) == 2
assert answer_negative_mark([1, 1]) == 2
assert answer_negative_mark([1, 2, 3]) == 4
# Bài 5
assert answer_find_disappeared([4, 3, 2, 7, 8, 2, 3, 1]) == [5, 6]
assert answer_find_disappeared([1, 1]) == [2]
assert answer_find_disappeared([1, 2, 3]) == []
# Bài 6
assert answer_find_duplicates([4, 3, 2, 7, 8, 2, 3, 1]) == [2, 3]
assert answer_find_duplicates([1, 1, 2]) == [1]
assert answer_find_duplicates([1]) == []

# Đối chiếu ngẫu nhiên với cách đơn giản
random.seed(448)
for _ in range(1000):
    n = random.randint(1, 9)
    arr = [random.randint(-2, 11) for _ in range(n)]
    assert answer_cyclic_sort(arr[:]) == answer_negative_mark(arr[:]) == first_missing_bruteforce(arr)
    arr2 = [random.randint(1, n) for _ in range(n)]
    assert answer_find_disappeared(arr2[:]) == [k for k in range(1, n + 1) if k not in arr2]
    base = list(range(1, n + 1))
    random.shuffle(base)
    twice = base[:random.randint(0, n // 2)]
    arr3 = base[:n - len(twice)] + twice        # dài đúng n, mỗi số xuất hiện 1 hoặc 2 lần
    random.shuffle(arr3)
    expected_dup = sorted(x for x in set(arr3) if arr3.count(x) == 2)
    assert answer_find_duplicates(arr3[:]) == expected_dup
print("Đáp án Bài 3-6 PASS toàn bộ test!")

Đáp án Bài 3-6 PASS toàn bộ test!


## 21. Template ngắn nên nhớ

```python
class Solution:
    def firstMissingPositive(self, nums: List[int]) -> int:
        n = len(nums)
        for i in range(n):                                   # đưa x về nhà x - 1
            while 1 <= nums[i] <= n and nums[nums[i] - 1] != nums[i]:
                j = nums[i] - 1                              # tính j TRƯỚC khi swap
                nums[i], nums[j] = nums[j], nums[i]
        for i in range(n):                                   # nhà đầu tiên sai chủ
            if nums[i] != i + 1:
                return i + 1
        return n + 1                                         # đủ 1..n
```

Câu thần chú: **"Đáp án trong [1, n+1] → mượn n ô làm n hộp → mỗi số về nhà x−1 → nhà sai chủ đầu tiên là đáp án."**

## 22. Checklist trước khi submit

- Mình có trả về `n + 1` khi mảng chứa đủ `1..n` chưa? (test `[1, 2, 3]`)
- Điều kiện `while` có kiểm tra **miền** `1 <= nums[i] <= n` **trước** khi dùng `nums[i] - 1` làm chỉ số chưa?
- Điều kiện `while` so sánh với **nhà đích** `nums[nums[i] - 1] != nums[i]` (chống lặp vô hạn khi số trùng) chưa? (test `[1, 1]`)
- Mình có tính `j = nums[i] - 1` ra biến riêng trước khi swap chưa?
- Nếu dùng dấu âm: rác thay bằng `n + 1` (không phải `0`), đọc bằng `abs`, đánh dấu bằng `-abs(...)`?
- Mình đã thử các edge case: `[1]`, `[2]`, `[-5]`, `[2147483647]`, mảng toàn số trùng?
- Mình hiểu và giải thích được vì sao tổng số swap `<= n` (mỗi swap khoá vĩnh viễn một ô)?